[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/razamoraz/FIsicaESTAdistica_2027_1/blob/main/Notebooks/Python/02_Camino_Aleatorio_1D_Reif.ipynb)

# 🎲 Laboratorio: Camino Aleatorio 1D, Valores Medios y Límite Gaussiano
**Curso:** Física Estadística (2027-1) — Facultad de Ciencias, UNAM  
**Profesor:** Dr. Roberto Antonio Zamora Zamora  
**Referencia Principal:** F. Reif, *Fundamentos de física estadística y térmica* (1968), Cap. 1 (Secciones 1.2, 1.3, 1.4 y 1.5).  

---

## 🎯 Objetivos de la Sesión
1. **Comprender la combinatoria fundamental** del camino aleatorio unidimensional y la deducción de la **distribución binomial** $W_N(n_1)$ y $P_N(m)$.
2. **Simular numéricamente ensambles estadísticos** de caminantes independientes utilizando `numpy` vectorizado.
3. **Verificar el cálculo analítico de momentos**: media $\bar{m} = N(p-q)$, dispersión $\overline{(\Delta m)^2} = 4Npq$ y la ley de fluctuaciones relativas $\Delta^* n_1 / \bar{n}_1 \propto 1/\sqrt{N}$.
4. **Analizar la transición al continuo para $N \gg 1$**: deducción por serie de Taylor de $\ln W$ y convergencia a la **distribución Gaussiana**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import comb, gammaln
from scipy.stats import norm

# Configuración estética de las gráficas
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.4
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["font.size"] = 11

print("Bibliotecas científicas importadas correctamente.")

---
## 1. El Problema del Camino Aleatorio en 1D (Reif 1.2)

Consideremos una partícula puntual que realiza pasos sucesivos de longitud $l$ a lo largo de una línea recta (una dimensión):
- Cada paso tiene probabilidad $p$ de ser hacia la **derecha** ($+l$) y probabilidad $q = 1 - p$ de ser hacia la **izquierda** ($-l$).
- Los pasos sucesivos son **estadísticamente independientes** entre sí.

Después de un total de $N$ pasos:
- $n_1$: número de pasos a la derecha.
- $n_2$: número de pasos a la izquierda ($n_2 = N - n_1$).

El desplazamiento neto respecto al origen (en unidades de longitud de paso $l$) es:
$$ m = n_1 - n_2 = n_1 - (N - n_1) = 2n_1 - N $$

> **Regla de paridad de Reif (Ec. 1.2.3):**  
> Como $m = 2n_1 - N$, si el número total de pasos $N$ es **par**, los posibles desplazamientos $m$ deben ser **pares**. Si $N$ es **impar**, $m$ debe ser **impar**.

La probabilidad de una secuencia particular que tenga $n_1$ pasos a la derecha y $n_2$ a la izquierda es $p^{n_1} q^{n_2}$. El número total de secuencias distintas con esta composición es el coeficiente binomial $\binom{N}{n_1} = \frac{N!}{n_1! (N - n_1)!}$.

Por lo tanto, la probabilidad $W_N(n_1)$ es la **distribución binomial**:
$$ W_N(n_1) = \frac{N!}{n_1! (N - n_1)!} p^{n_1} q^{N - n_1} $$

Dado que $n_1 = \frac{1}{2}(N + m)$, la probabilidad de encontrar la partícula en la posición $m$ tras $N$ pasos es:
$$ P_N(m) = W_N\left(\frac{N + m}{2}\right) = \frac{N!}{\left(\frac{N+m}{2}\right)! \left(\frac{N-m}{2}\right)!} p^{(N+m)/2} (1-p)^{(N-m)/2} $$

In [ ]:
def binomial_prob(N, n1, p=0.5):
    """
    Calcula W_N(n1) de forma numéricamente estable usando log-factoriales (gammaln).
    """
    n1 = np.asarray(n1, dtype=int)
    q = 1.0 - p
    valid = (n1 >= 0) & (n1 <= N)
    w = np.zeros_like(n1, dtype=float)
    
    # Logaritmo de la distribución binomial: ln(N!) - ln(n1!) - ln((N-n1)!) + n1*ln(p) + (N-n1)*ln(q)
    log_w = (gammaln(N + 1) - gammaln(n1[valid] + 1) - gammaln(N - n1[valid] + 1) 
             + n1[valid] * np.log(p) + (N - n1[valid]) * np.log(q))
    w[valid] = np.exp(log_w)
    return w

def prob_displacement(N, m, p=0.5):
    """
    Calcula P_N(m) verificando la regla de paridad (N + m debe ser par y |m| <= N).
    """
    m = np.asarray(m, dtype=int)
    valid_parity = ((N + m) % 2 == 0) & (np.abs(m) <= N)
    p_m = np.zeros_like(m, dtype=float)
    
    n1 = (N + m[valid_parity]) // 2
    p_m[valid_parity] = binomial_prob(N, n1, p)
    return p_m

# Ejemplo didáctico de Reif: N = 3 pasos, p = q = 0.5
print("=== Ejemplo de Reif (N = 3, p = 0.5) ===")
m_vals = np.array([-3, -2, -1, 0, 1, 2, 3])
probs = prob_displacement(3, m_vals, p=0.5)
for m_val, pr in zip(m_vals, probs):
    if pr > 0:
        n1_val = (3 + m_val) // 2
        print(f"  m = {m_val:+2d} (n1 = {n1_val}, n2 = {3-n1_val}) -> P_3({m_val:+2d}) = {pr:.4f} ({int(round(pr*8))}/8)")
    else:
        print(f"  m = {m_val:+2d} -> Imposible por paridad (P = 0)")

In [ ]:
# Reproducción de las Figuras 1.2.3 y 1.4.1 de Reif para N = 20 pasos
N = 20
n1_range = np.arange(0, N + 1)
m_range = 2 * n1_range - N

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))

# Fig 1.2.3: Caso simétrico p = q = 0.5
p_sym = 0.5
w_sym = binomial_prob(N, n1_range, p=p_sym)
ax1.stem(m_range, w_sym, linefmt="b-", markerfmt="bo", basefmt="k-")
ax1.set_title(f"Fig. 1.2.3 de Reif: Simétrico ($N={N}, p={p_sym}$)")
ax1.set_xlabel("Desplazamiento neto $m$")
ax1.set_ylabel("Probabilidad $P_N(m) = W_N(n_1)$")
ax1.set_xticks(m_range[::2])
ax1.axvline(0, color="red", linestyle="--", alpha=0.7, label="Media $\\bar{m} = 0$")
ax1.legend()

# Fig 1.4.1: Caso asimétrico p = 0.6, q = 0.4
p_asym = 0.6
w_asym = binomial_prob(N, n1_range, p=p_asym)
ax2.stem(m_range, w_asym, linefmt="teal", markerfmt="o", basefmt="k-")
mean_m_asym = N * (p_asym - (1 - p_asym))
ax2.set_title(f"Fig. 1.4.1 de Reif: Asimétrico ($N={N}, p={p_asym}, q=0.4$)")
ax2.set_xlabel("Desplazamiento neto $m$")
ax2.set_ylabel("Probabilidad $P_N(m) = W_N(n_1)$")
ax2.set_xticks(m_range[::2])
ax2.axvline(mean_m_asym, color="darkorange", linestyle="--", alpha=0.9, 
            label=f"Media $\\bar{{m}} = {mean_m_asym:.1f}$")
ax2.legend()

plt.tight_layout()
plt.show()

---
## 2. Simulación Ensamble Monte Carlo (NumPy Vectorizado)

En física estadística, las probabilidades se interpretan operacionalmente como promedios sobre un **ensamble estadístico** de $M$ sistemas independientes idénticamente preparados.

Podemos simular $M$ caminantes simultáneamente en una matriz bidimensional $S \in \{-1, +1\}^{M \times N}$:
- Cada fila representa la historia temporal de un caminante.
- La posición tras $t$ pasos se obtiene acumulando los desplazamientos con `np.cumsum`:
$$ x_k(t) = \sum_{j=1}^t s_{k, j}, \quad s_{k, j} \in \{-1, +1\} $$

In [ ]:
def simulate_random_walks(M_walkers, N_steps, p=0.5, seed=42):
    """
    Simula vectorialmente M caminantes aleatorios de N pasos.
    Retorna:
      trajectories: matriz de dimensiones (M, N + 1) con la posición x(t) desde t=0 hasta N.
    """
    rng = np.random.default_rng(seed)
    # Genera pasos: +1 con prob p, -1 con prob (1-p)
    random_draws = rng.random(size=(M_walkers, N_steps))
    steps = np.where(random_draws < p, 1, -1)
    
    # Posición inicial x(0) = 0 para todos los caminantes
    initial_pos = np.zeros((M_walkers, 1), dtype=int)
    trajectories = np.hstack([initial_pos, np.cumsum(steps, axis=1)])
    return trajectories

# Simulación de un ensamble grande
M_walkers = 25000
N_steps = 50
p_bias = 0.5

trajectories = simulate_random_walks(M_walkers, N_steps, p=p_bias)
final_positions = trajectories[:, -1]

print(f"Simulación completada con {M_walkers} caminantes de {N_steps} pasos.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5.5))

# Panel A: Trayectorias individuales en el tiempo
t_axis = np.arange(N_steps + 1)
sample_trajectories = trajectories[:25] # Primeras 25 trayectorias
for traj in sample_trajectories:
    ax1.plot(t_axis, traj, alpha=0.5, linewidth=1.2)

# Curva de desviación estándar teórica +/- sqrt(t)
sigma_t = np.sqrt(4 * t_axis * p_bias * (1 - p_bias))
ax1.plot(t_axis, sigma_t, "k--", linewidth=2, label=r"$+\Delta^* m(t) = \sqrt{t}$")
ax1.plot(t_axis, -sigma_t, "k--", linewidth=2, label=r"$-\Delta^* m(t) = -\sqrt{t}$")
ax1.set_title("A. Trayectorias individuales $x(t)$ vs. Tiempo")
ax1.set_xlabel("Número de pasos $t$")
ax1.set_ylabel("Posición $x(t)$")
ax1.legend(loc="upper left")

# Panel B: Histograma del ensamble al paso final vs. Distribución Binomial Teórica
bins = np.arange(-N_steps - 1, N_steps + 3, 2) - 0.5
counts, bin_edges = np.histogram(final_positions, bins=bins, density=True)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# Probabilidad teórica para valores permitidos de m
m_theory = np.arange(-N_steps, N_steps + 1, 2)
p_theory = prob_displacement(N_steps, m_theory, p=p_bias)

# Multiplicamos por 2 para comparar con la densidad continua debido al espaciamiento Delta m = 2
ax2.bar(bin_centers, counts * 2, width=1.6, color="royalblue", alpha=0.6, 
        label=f"Ensamble Simulado ($M={M_walkers}$)")
ax2.plot(m_theory, p_theory, "ro", markersize=6, label="Teoría Binomial $P_N(m)$")
ax2.set_title(f"B. Distribución final al paso $N = {N_steps}$")
ax2.set_xlabel("Desplazamiento final $m$")
ax2.set_ylabel("Probabilidad $P_N(m)$")
ax2.legend()

plt.tight_layout()
plt.show()

---
## 3. Valores Medios, Momentos Estadísticos y Fluctuaciones (Reif 1.3 y 1.4)

### 3.1 Definición General de Momentos (Reif 1.3)
Dada una variable discreta $u$ con distribución de probabilidad normalizada $\sum_i P(u_i) = 1$:
- **Valor medio:** $\bar{u} = \sum_i P(u_i) u_i$
- **Desviación respecto a la media:** $\Delta u = u - \bar{u}$, con $\overline{\Delta u} = 0$.
- **Dispersión (segundo momento central / varianza):**
$$ \overline{(\Delta u)^2} = \sum_i P(u_i) (u_i - \bar{u})^2 = \overline{u^2} - \bar{u}^2 \ge 0 $$
- **Desviación cuadrática media (RMS):** $\Delta^* u = \sqrt{\overline{(\Delta u)^2}}$.

---

### 3.2 Cálculo Analítico mediante el Truco de Diferenciación de Reif (Reif 1.4)
Para evaluar $\bar{n}_1 = \sum_{n_1=0}^N \binom{N}{n_1} p^{n_1} q^{N-n_1} n_1$, Reif introduce el truco de considerar $p$ y $q$ como variables independientes:
$$ n_1 p^{n_1} = p \frac{\partial}{\partial p}(p^{n_1}) $$

Intercambiando la suma y la derivada:
$$ \bar{n}_1 = p \frac{\partial}{\partial p} \sum_{n_1=0}^N \binom{N}{n_1} p^{n_1} q^{N-n_1} = p \frac{\partial}{\partial p}(p+q)^N = p N(p+q)^{N-1} $$
Evaluando para $p + q = 1$:
$$ \bar{n}_1 = N p, \qquad \bar{n}_2 = N q $$

Para el desplazamiento neto $m = n_1 - n_2$:
$$ \bar{m} = \bar{n}_1 - \bar{n}_2 = N(p - q) $$

Aplicando el operador $\left(p \frac{\partial}{\partial p}\right)^2$ se obtiene la dispersión:
$$ \overline{(\Delta n_1)^2} = N p q $$
$$ \overline{(\Delta m)^2} = 4 \overline{(\Delta n_1)^2} = 4 N p q $$

> **Caso simétrico ($p = q = 1/2$):**  
> - $\bar{m} = 0$  
> - $\overline{(\Delta m)^2} = N \implies \Delta^* m = \sqrt{N}$

---

### 3.3 Ancho Relativo y Límite Termodinámico ($1/\sqrt{N}$)
Una cantidad crucial para comprender el origen de la certeza macroscópica en física estadística es el **ancho relativo** de la distribución:
$$ \frac{\Delta^* n_1}{\bar{n}_1} = \frac{\sqrt{Npq}}{Np} = \sqrt{\frac{q}{p}} \frac{1}{\sqrt{N}} $$

A medida que $N \to \infty$, aunque el ancho absoluto $\Delta^* n_1 \propto \sqrt{N}$ crece, **las fluctuaciones relativas decrecen como $N^{-1/2}$**. Para un número macroscópico de partículas ($N \sim 10^{23}$), las fluctuaciones relativas son del orden de $10^{-11.5}$, haciendo que la variable se comporte de forma prácticamente determinista.

In [ ]:
# Verificación numérica de los momentos y del escalamiento con N
N_values = np.unique(np.logspace(1, 3.3, 20).astype(int))
M_test = 5000
p_test = 0.5
q_test = 1.0 - p_test

mean_m_emp = []
rms_m_emp = []
rel_fluct_n1_emp = []

rng = np.random.default_rng(123)
for N_val in N_values:
    draws = rng.random(size=(M_test, N_val))
    steps = np.where(draws < p_test, 1, -1)
    m_final = np.sum(steps, axis=1)
    n1_final = (N_val + m_final) / 2.0
    
    mean_m_emp.append(np.mean(m_final))
    rms_m_emp.append(np.std(m_final))
    rel_fluct_n1_emp.append(np.std(n1_final) / np.mean(n1_final))

mean_m_emp = np.array(mean_m_emp)
rms_m_emp = np.array(rms_m_emp)
rel_fluct_n1_emp = np.array(rel_fluct_n1_emp)

print("Cálculo de momentos para diferentes valores de N completado.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Panel A: Dispersión absoluta Delta* m vs sqrt(N)
theory_rms = np.sqrt(4 * N_values * p_test * q_test)
ax1.plot(N_values, theory_rms, "r-", linewidth=2, label=r"Teoría: $\Delta^* m = \sqrt{4Npq} = \sqrt{N}$")
ax1.scatter(N_values, rms_m_emp, color="navy", zorder=3, label=f"Simulación Monte Carlo ($M={M_test}$)")
ax1.set_title(r"A. Crecimiento del ancho absoluto: $\Delta^* m \propto \sqrt{N}$")
ax1.set_xlabel("Número de pasos $N$")
ax1.set_ylabel(r"Desviación RMS $\Delta^* m$")
ax1.legend()

# Panel B: Fluctuación relativa Delta* n1 / n1_bar vs 1/sqrt(N) en escala Log-Log
theory_rel = np.sqrt(q_test / p_test) / np.sqrt(N_values)
ax2.loglog(N_values, theory_rel, "r-", linewidth=2, label=r"Teoría: $\frac{\Delta^* n_1}{\bar{n}_1} = \frac{1}{\sqrt{N}}$")
ax2.scatter(N_values, rel_fluct_n1_emp, color="darkgreen", zorder=3, label="Simulación")
ax2.set_title(r"B. Decaimiento de fluctuaciones relativas: $\propto N^{-1/2}$")
ax2.set_xlabel("Número de pasos $N$")
ax2.set_ylabel(r"Ancho relativo $\Delta^* n_1 / \bar{n}_1$")
ax2.legend()

plt.tight_layout()
plt.show()

---
## 4. Límite para $N \gg 1$ y Aproximación Gaussiana (Reif 1.5)

### 4.1 ¿Por qué expandir $\ln W$ en lugar de $W$?
Cuando $N$ es grande, $W(n_1)$ presenta un pico sumamente estrecho y pronunciado alrededor de su valor máximo $\tilde{n}_1 = Np$. Una expansión directa de Taylor de $W(n_1)$ requeriría infinidad de términos y fallaría en converger. En contraste, **$\ln W(n_1)$ varía con extrema lentitud**, lo que hace que su serie de Taylor converja rápidamente.

Definiendo la desviación $\eta = n_1 - \tilde{n}_1$:
$$ \ln W(n_1) = \ln W(\tilde{n}_1) + B_1 \eta + \frac{1}{2} B_2 \eta^2 + \frac{1}{6} B_3 \eta^3 + \dots $$
donde $B_k = \left. \frac{d^k \ln W}{d n_1^k} \right|_{n_1 = \tilde{n}_1}$.

### 4.2 Evaluación de los Coeficientes mediante Stirling
Utilizando la aproximación continua $\frac{d \ln n!}{dn} \approx \ln n$:
1. **Primer derivada (condición de máximo):**
   $$ \frac{d \ln W}{d n_1} = -\ln n_1 + \ln(N - n_1) + \ln p - \ln q = 0 \implies \tilde{n}_1 = N p $$
   Por lo tanto, $B_1 = 0$.

2. **Segunda derivada (curvatura):**
   $$ B_2 = \left. \frac{d^2 \ln W}{d n_1^2} \right|_{\tilde{n}_1} = -\frac{1}{Np} - \frac{1}{Nq} = -\frac{1}{N p q} < 0 $$

3. **Términos de orden superior:**
   $$ |B_3| \approx \mathcal{O}\left(\frac{1}{N^2}\right), \quad |B_4| \approx \mathcal{O}\left(\frac{1}{N^3}\right) $$
   Para $|\eta| \ll Npq$, los términos cúbicos y superiores son despreciables.

Tratando $n_1$ como variable continua y normalizando $\int_{-\infty}^\infty W(n_1) dn_1 = 1$, obtenemos la **distribución Gaussiana**:
$$ W(n_1) \approx \frac{1}{\sqrt{2\pi N p q}} \exp\left( -\frac{(n_1 - N p)^2}{2 N p q} \right) $$

En términos del desplazamiento neto $m = 2n_1 - N$ (con $dm = 2 dn_1$ y $\sigma_m = \sqrt{4Npq}$):
$$ \mathcal{P}(m) = \frac{1}{\sqrt{2\pi \sigma_m^2}} \exp\left( -\frac{(m - \bar{m})^2}{2 \sigma_m^2} \right) $$

Para el caso simétrico ($p = q = 1/2$):
$$ \mathcal{P}(m) = \frac{1}{\sqrt{2\pi N}} \exp\left( -\frac{m^2}{2N} \right) $$

In [ ]:
# Comparación sistemática Binomial vs. Gaussiana para diferentes valores de N
test_Ns = [6, 20, 60, 200]
p_val = 0.5

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for idx, N_curr in enumerate(test_Ns):
    ax = axes[idx]
    
    # Valores discretos permitidos de m
    m_disc = np.arange(-N_curr, N_curr + 1, 2)
    w_binom = prob_displacement(N_curr, m_disc, p=p_val)
    
    # Distribución continua gaussiana evaluada sobre m
    # Nótese: Como m salta de 2 en 2, la masa de probabilidad discreta es ~ 2 * P_gauss(m)
    sigma_m = np.sqrt(4 * N_curr * p_val * (1 - p_val))
    mu_m = N_curr * (p_val - (1 - p_val))
    
    m_cont = np.linspace(-N_curr, N_curr, 500)
    p_gauss_cont = 2.0 * norm.pdf(m_cont, loc=mu_m, scale=sigma_m)
    
    # Gráfico
    ax.stem(m_disc, w_binom, linefmt="b-", markerfmt="bo", basefmt="k-", 
            label="Binomial Exacta $P_N(m)$")
    ax.plot(m_cont, p_gauss_cont, "r-", linewidth=2.2, 
            label=f"Aprox. Gaussiana ($2 \\times \\mathcal{{N}}(0, \\sigma={sigma_m:.1f})$)")
    
    ax.set_title(f"N = {N_curr} pasos ($Npq = {N_curr * p_val * (1-p_val):.1f}$)")
    ax.set_xlabel("Desplazamiento $m$")
    ax.set_ylabel("Probabilidad")
    ax.set_xlim(-3.2 * sigma_m, 3.2 * sigma_m)
    ax.legend(loc="upper right", fontsize=10)

plt.tight_layout()
plt.show()

---
## 5. Actividades Prácticas y Preguntas Conceptuales

### 🧠 Actividad 1: El Límite de Detección de una Deriva Débil (*Biased Drift*)
Imagina que un caminante tiene un ligero sesgo hacia la derecha ($p = 0.52$).  
¿Cuántos pasos $N$ se requieren para que el desplazamiento medio $\bar{m} = N(p-q)$ supere con creces el ruido fluctuante $\Delta^* m = \sqrt{4Npq}$? (Criterio de detección: $\bar{m} > 3 \Delta^* m$).

In [ ]:
# Solución analítica y numérica a la Actividad 1
p_bias = 0.52
q_bias = 1.0 - p_bias

# Condición: N(p - q) > 3 * sqrt(4 N p q)
# N^2 (p - q)^2 > 9 * 4 N p q  ==>  N > 36 * p * q / (p - q)^2
N_threshold = (36.0 * p_bias * q_bias) / ((p_bias - q_bias) ** 2)
print(f"=== Límite de Detección de Deriva (p = {p_bias}) ===")
print(f"Número mínimo de pasos necesarios para señal/ruido > 3: N >= {int(np.ceil(N_threshold))} pasos.")

# Demostración con Monte Carlo
N_test_steps = int(np.ceil(N_threshold))
trajs_drift = simulate_random_walks(10000, N_test_steps, p=p_bias, seed=99)
pos_final = trajs_drift[:, -1]

print(f"Media observada: {np.mean(pos_final):.2f} (Teoría: {N_test_steps*(p_bias-q_bias):.2f})")
print(f"RMS observado: {np.std(pos_final):.2f} (Teoría: {np.sqrt(4*N_test_steps*p_bias*q_bias):.2f})")
print(f"Relación Señal/Ruido (SNR): {np.mean(pos_final)/np.std(pos_final):.2f}")

### 🧠 Actividad 2: Tiempo Medio de Escape y Relación $t \propto L^2$
Si colocamos dos barreras absorbentes en $x = -L$ y $x = +L$, ¿cómo escala el tiempo medio de escape $\langle \tau \rangle$ con la distancia $L$?
Como $\Delta^* m \propto \sqrt{t}$, para alcanzar una distancia $L$ se requieren en promedio $t \sim L^2$ pasos. Esta es la firma característica de los **procesos difusivos clásicos** ($x_{\text{rms}} = \sqrt{2 D t}$), a diferencia del transporte balístico ($x = v t$).

In [ ]:
# Simulación del tiempo de escape a barreras en +/- L
L_values = np.array([5, 10, 15, 20, 25, 30])
n_trials = 2000
mean_escape_times = []

rng = np.random.default_rng(42)
for L in L_values:
    escape_times = []
    for _ in range(n_trials):
        pos = 0
        steps = 0
        while abs(pos) < L:
            step = 1 if rng.random() < 0.5 else -1
            pos += step
            steps += 1
        escape_times.append(steps)
    mean_escape_times.append(np.mean(escape_times))

mean_escape_times = np.array(mean_escape_times)

# Gráfica de escalamiento
plt.figure(figsize=(8, 5))
plt.plot(L_values, L_values**2, "r--", linewidth=2, label=r"Teoría Difusiva: $\\langle t \\rangle = L^2$")
plt.scatter(L_values, mean_escape_times, color="darkviolet", s=60, zorder=3, label="Simulación Monte Carlo")
plt.title(r"Tiempo Medio de Escape vs. Tamaño de Barrera $L$")
plt.xlabel(r"Distancia a la barrera $L$")
plt.ylabel(r"Tiempo medio de escape $\\langle \\tau \\rangle$ (pasos)")
plt.legend()
plt.show()

---
## 📌 Conclusiones y Conexión con las Próximas Sesiones
1. **Microestados vs. Macroestados:** A través de la combinatoria $\binom{N}{n_1}$, vimos que la inmensa mayoría de las trayectorias microscópicas dan lugar a un desplazamiento neto $m$ cercano a la media $\bar{m} = N(p-q)$.
2. **El límite termodinámico:** Las fluctuaciones relativas decrecen como $1/\sqrt{N}$, lo que explica por qué los sistemas macroscópicos con $N \sim 10^{23}$ parecen seguir leyes deterministas a pesar de ser estocásticos a nivel microscópico.
3. **Transición al continuo y Difusión:** En el límite $N \gg 1$ y pasos infinitamente pequeños ($\Delta t \to 0, \Delta x \to 0$), la distribución binomial se convierte en la solución fundamental (núcleo Gaussiano) de la **Ecuación de Difusión**:
   $$ \frac{\partial P(x, t)}{\partial t} = D \frac{\partial^2 P(x, t)}{\partial x^2}, \qquad D = \frac{l^2}{2\tau} $$
   lo cual exploraremos en detalle en las **Sesiones 5 y 6**.